In [1]:
# Run in a Colab cell. On a Mac terminal, drop the "!" and use curl -L -o PBC.zip "..." and shasum -a 256 PBC.zip
!wget -q -O PBC.zip "https://data.mendeley.com/public-files/datasets/snkd93bnjr/files/2fc38728-2ae7-4a62-a857-032af82334c3/file_downloaded"
!sha256sum PBC.zip
# expected: 6c50b82d1ab499b0b5a6beda0d0f75e6013df4a7233f116da354351ac10ca4f8

6c50b82d1ab499b0b5a6beda0d0f75e6013df4a7233f116da354351ac10ca4f8  PBC.zip


In [2]:
!rm -rf data/PBC_dataset_normal_DIB
!rm -rf data/__MACOSX
!mkdir -p data

In [3]:
!unzip -q PBC.zip -x "__MACOSX/*" -d data/

In [4]:
from torchvision.datasets import ImageFolder

ds = ImageFolder("data/PBC_dataset_normal_DIB")
print(len(ds))      # 17093  (see note below)
print(ds.classes)   # ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

17093
['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']


In [5]:
!ls data/

PBC_dataset_normal_DIB


In [7]:
import matplotlib.pyplot as plt
from PIL import Image
import os

hidden_file_path = "data/PBC_dataset_normal_DIB/neutrophil/.DS_169665.jpg"

if os.path.exists(hidden_file_path):
    img = Image.open(hidden_file_path)
    plt.imshow(img)
    plt.title("The Hidden Image: .DS_169665.jpg")
    plt.axis('off')
    plt.show()
else:
    print("Could not find the file. Are you sure the dataset is unzipped in 'data/'?")

UnidentifiedImageError: cannot identify image file 'data/PBC_dataset_normal_DIB/neutrophil/.DS_169665.jpg'

In [8]:
import os

bad_file = "data/PBC_dataset_normal_DIB/neutrophil/.DS_169665.jpg"

if os.path.exists(bad_file):
    os.remove(bad_file)

print("Removed invalid hidden file:", not os.path.exists(bad_file))

Removed invalid hidden file: True


In [9]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision.datasets import ImageFolder

dataset_root = "data/PBC_dataset_normal_DIB"

# Load cleaned dataset
ds = ImageFolder(dataset_root)

print("Total valid images:", len(ds))
print("Classes:", ds.classes)

# Store RELATIVE paths instead of Colab-specific paths
relative_paths = [
    os.path.relpath(path, dataset_root)
    for path, _ in ds.samples
]

labels = [label for _, label in ds.samples]

# 70% train, 30% temporary
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    relative_paths,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

# Split remaining 30% equally -> 15% validation, 15% test
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

# Build dataframes
df_train = pd.DataFrame({
    "file_path": train_paths,
    "label_idx": train_labels,
    "split": "train"
})

df_val = pd.DataFrame({
    "file_path": val_paths,
    "label_idx": val_labels,
    "split": "val"
})

df_test = pd.DataFrame({
    "file_path": test_paths,
    "label_idx": test_labels,
    "split": "test"
})

df_master = pd.concat(
    [df_train, df_val, df_test],
    ignore_index=True
)

# Add readable class names
idx_to_class = {v: k for k, v in ds.class_to_idx.items()}
df_master["class_name"] = df_master["label_idx"].map(idx_to_class)

# Save
df_master.to_csv("pbc_master_split.csv", index=False)

print("\nMaster split created.")
print(df_master["split"].value_counts())

print("\nClass counts by split:")
print(
    pd.crosstab(
        df_master["class_name"],
        df_master["split"]
    )
)

Total valid images: 17092
Classes: ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

Master split created.
split
train    11964
val       2564
test      2564
Name: count, dtype: int64

Class counts by split:
split         test  train  val
class_name                    
basophil       183    853  182
eosinophil     467   2182  468
erythroblast   233   1086  232
ig             434   2026  435
lymphocyte     182    850  182
monocyte       213    994  213
neutrophil     500   2330  499
platelet       352   1643  353


In [10]:
print("Total rows:", len(df_master))
print("Duplicate paths:", df_master["file_path"].duplicated().sum())
print("Missing class names:", df_master["class_name"].isna().sum())

Total rows: 17092
Duplicate paths: 0
Missing class names: 0


In [11]:
from google.colab import drive
drive.mount('/content/drive')

df_master.to_csv(
    "/content/drive/MyDrive/Hemavision/pbc_master_split.csv",
    index=False
)

Mounted at /content/drive


In [12]:
!sha256sum pbc_master_split.csv

909fcfc51a6d6c630e6e710378f715d3ccdaf13b601ab92afdfa26757f9201fe  pbc_master_split.csv
